# 03 — Drift Check (PSI)

PSI is a simple distribution shift signal.

In [ ]:
import numpy as np
import pandas as pd

def psi(expected: pd.Series, actual: pd.Series, buckets=10):
    expected = expected.dropna().astype(float)
    actual = actual.dropna().astype(float)
    q = np.linspace(0,1,buckets+1)
    breaks = np.unique(np.quantile(expected,q))
    if len(breaks)<3: return 0.0

    def pct(s):
        c,_ = np.histogram(s,bins=breaks)
        p = c/max(1,c.sum())
        return np.clip(p,1e-6,1.0)

    e = pct(expected); a = pct(actual)
    return float(np.sum((a-e)*np.log(a/e)))

df = pd.read_csv("data/transactions.csv").sort_values("ts").reset_index(drop=True)
cut = int(len(df)*0.7)
psi(df.iloc[:cut]["amount"], df.iloc[cut:]["amount"])
